<a href="https://colab.research.google.com/github/GuilleeSS/MyRepo/blob/main/nmlab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import scipy
import scipy.linalg
import numpy as np
import copy

# 1. Definición de la función linearsolver del laboratorio (Faltaba esto en tu celda)
def linearsolver(A, b):
    n = len(A)
    M = A

    i = 0
    for x in M:
        x.append(b[i])
        i += 1

    for k in range(n):
        # Pivoteo
        for i in range(k, n):
            if abs(M[i][k]) > abs(M[k][k]):
                M[k], M[i] = M[i], M[k]
            else:
                pass

        # Eliminación Gaussiana
        for j in range(k+1, n):
            q = float(M[j][k]) / M[k][k]
            for m in range(k, n+1):
                M[j][m] -= q * M[k][m]

    x = [0 for i in range(n)]

    # Sustitución hacia atrás para calcular x
    x[n-1] = float(M[n-1][n]) / M[n-1][n-1]
    for i in range(n-1, -1, -1):
        z = 0
        for j in range(i+1, n):
            z = z + float(M[i][j]) * x[j]
        x[i] = float(M[i][n] - z) / M[i][i]
    return x


# 2. Tu entrega: Definición del sistema de ecuaciones del Ejercicio 2
# Modificamos los datos para resolver el problema solicitado en la guía
A = [[4, -1, 1],
     [2, 5, 2],
     [1, 2, 4]]
b = [8, 3, 11]

# Hacemos una copia profunda para no alterar la matriz A original
A1 = copy.deepcopy(A)

# Ejecución con el algoritmo manual del laboratorio
x_manual = linearsolver(A1, b)

print(" result with LINEARSOLVER")
print("Solution:", x_manual)

# Ejecución con la librería Scipy para comparar
x_scipy = scipy.linalg.solve(A, b)

print("\n result with LINALG.SOLVE (SCIPY)")
print("Solution:", x_scipy)

 result with LINEARSOLVER
Solution: [1.0, -1.0, 3.0]

 result with LINALG.SOLVE (SCIPY)
Solution: [ 1. -1.  3.]


In [18]:
import numpy as np
import scipy.linalg

# 1. Tu función base de descomposición provista en el lab
def LU_partial_decomposition(matrix):
    n, m = matrix.shape
    P = np.identity(n)
    L = np.identity(n)
    U = matrix.copy()
    PF = np.identity(n)
    LF = np.zeros((n,n))
    for k in range(0, n - 1):
        index = np.argmax(abs(U[k:,k]))
        index = index + k
        if index != k:
            P = np.identity(n)
            P[[index,k],k:n] = P[[k,index],k:n]
            U[[index,k],k:n] = U[[k,index],k:n]
            PF = np.dot(P,PF)
            LF = np.dot(P,LF)
        L = np.identity(n)
        for j in range(k+1,n):
            L[j,k] = -(U[j,k] / U[k,k])
            LF[j,k] = (U[j,k] / U[k,k])
        U = np.dot(L,U)
    np.fill_diagonal(LF, 1)
    return PF, LF, U

# 2. EXTENSIÓN: Funciones para resolver mediante sustitución
def forward_substitution(L, b):
    # Resuelve L y = b
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        sum_ly = sum(L[i][j] * y[j] for j in range(i))
        y[i] = b[i] - sum_ly
    return y

def backward_substitution(U, y):
    # Resuelve U x = y
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        sum_ux = sum(U[i][j] * x[j] for j in range(i + 1, n))
        x[i] = (y[i] - sum_ux) / U[i][i]
    return x

def lu_solver(A, b):
    # Resuelve Ax = b -> P L U x = b -> L U x = P b
    P, L, U = LU_partial_decomposition(A)
    # Aplicar permutación al vector b
    Pb = np.dot(P, b)
    # 1. Resolver L y = P b
    y = forward_substitution(L, Pb)
    # 2. Resolver U x = y
    x = backward_substitution(U, y)
    return x

# 3. Resolver el ejercicio sugerido de la clase
A_clase = np.array([[1, 2, 1],
                    [1, -2, 2],
                    [2, 12, -2]], dtype=float)
b_clase = np.array([0, 4, 4], dtype=float)

# Solución con nuestra extensión LU
x_lu = lu_solver(A_clase, b_clase)

print("system resolution (LU)")
print("Matrix A:\n", A_clase)
print("Vector b:", b_clase)
print("\nSolution:", x_lu)

# Verificación opcional con Scipy
x_scipy_lu = scipy.linalg.solve(A_clase, b_clase)
print("Scipy verification:", x_scipy_lu)

system resolution (LU)
Matrix A:
 [[ 1.  2.  1.]
 [ 1. -2.  2.]
 [ 2. 12. -2.]]
Vector b: [0. 4. 4.]

Solution: [11.  -2.5 -6. ]
Scipy verification: [11.  -2.5 -6. ]


In [19]:
import numpy as np

# 1. Definición de algoritmos base del laboratorio
def iterative_newton(fun, x_init, jacobian):
    max_iter = 150
    epsilon = 1e-8
    x_last = x_init
    for k in range(max_iter):
        J = np.array(jacobian(x_last))
        F = np.array(fun(x_last))
        Jinv = np.linalg.inv(J)
        diff = np.dot(Jinv, F)
        x_last = x_last - diff
        if np.linalg.norm(diff) < epsilon:
            print('Convergence!, number of iterations:', k)
            break
    else:
        print('No converged')
    return x_last

def iterative_aitken(fun, x_init, jacobian):
    max_iter = 50
    epsilon = 1e-8
    loop_break = 0
    l = len(x_init)
    X = np.zeros([max_iter+l+2, l])
    X[0] = x_init

    for n in range(0, max_iter, l):
        if loop_break:
            break
        for k in range(0, l):
            J = np.array(jacobian(X[n+k]))
            F = np.array(fun(X[n+k]))
            Jinv = np.linalg.inv(J)
            diff = np.dot(Jinv, F)
            X[n+k+1] = X[n+k] - diff

        X1 = np.zeros([l, l])
        X2 = np.zeros([l, l])
        X3 = np.zeros([l, l])

        for i in range(l):
            for j in range(l):
                X1[j, i] = X[n+k-1-i, j]
                X2[j, i] = X[n+k-i, j]
                X3[j, i] = X[n+k+1-i, j]

        if abs(np.linalg.det(np.subtract(X2, X1))) > 1e-16:
            Lambda = np.dot(np.subtract(X3, X2), np.linalg.inv(np.subtract(X2, X1)))
            XX1 = np.linalg.inv(np.subtract(np.identity(l), Lambda))
            XX2 = np.subtract(X[n+1], np.dot(Lambda, X[n]))
            XX = np.dot(XX1, XX2)
            print(" Without accelerating = ", X[n+k+1])
            print(" Aitken accelerated = ", XX)
            X[n+k+1] = XX

        if np.linalg.norm(diff) < epsilon:
            print('Convergence with Aitken!, number of iterations:', n)
            X_out = X[n+k+1]
            loop_break = 1
            break
    else:
        print('No converged')
        X_out = X[n+k+1]
    return X_out


# 2. El nuevo ejercicio lento que propusimos
def function_slow_exercise(xy):
    x, y = xy
    return [x**3 - y, x - y**3]

def jacobian_slow_exercise(xy):
    x, y = xy
    return [[3*x**2, -1],
            [1, -3*y**2]]


# 3. Ejecuciones de prueba
print("NEWTON STANDARD")
x_sol_normal = iterative_newton(function_slow_exercise, [0.5, 0.5], jacobian_slow_exercise)
print('standard solution:', x_sol_normal)
print(" ")

print("AITKEN AACCELERATION")
x_sol_aitken = iterative_aitken(function_slow_exercise, [0.5, 0.5], jacobian_slow_exercise)
print('Aitken solution:', x_sol_aitken)

NEWTON STANDARD
Convergence!, number of iterations: 1
standard solution: [-1. -1.]
 
AITKEN AACCELERATION
Convergence with Aitken!, number of iterations: 0
Aitken solution: [-1. -1.]
